# Replay a recording (M1.3)

The engine runs headless and emits a Parquet **recording**; this notebook reads it back and reconstructs train positions over time.

First generate a recording (from the repo root):

```bash
uv run dbsim run --date 20260616 --db data/processed/gtfs-fv.duckdb \
    --all --delay 124021:0:1200 --record data/processed/run-20260616.parquet
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from dbsim.model import Timetable, format_hms
from dbsim.record import load_recording

REC = Path("../data/processed/run-20260616.parquet")
DB = Path("../data/processed/gtfs-fv.duckdb")

rec = load_recording(REC)
rec.meta

## Network activity over the day

How many trains are underway (departed, not yet arrived) at each minute.

In [ ]:
times = list(range(4 * 3600, 26 * 3600, 300))  # every 5 min, 04:00–26:00
active = [rec.active_trip_count(t) for t in times]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot([t / 3600 for t in times], active, color="#1f77b4")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Trains underway")
ax.set_title(f"Network activity — {rec.meta.service_date}")
ax.grid(color="0.92")
plt.show()

## One train's trajectory

ICE 22 (trip `124021`) was given a +20 min primary delay at Frankfurt; read its movement events back from the recording.

In [ ]:
with Timetable(DB) as tt:
    rows = tt.connection.execute("SELECT stop_id, stop_name FROM stops").fetchall()
    coords = tt.connection.execute("SELECT stop_id, stop_lat, stop_lon FROM stops").fetchall()
names = {str(r[0]): str(r[1]) for r in rows}
coord_map = {str(r[0]): (r[1], r[2]) for r in coords if r[1] is not None}

for ev in rec.train_events("124021"):
    dev = ev.deviation_s or 0
    tag = f" (+{dev // 60} min)" if dev else ""
    print(f"{format_hms(ev.time_s)}  {ev.event:<7} {names.get(ev.stop_id, ev.stop_id)}{tag}")

## A replay snapshot

Interpolated train positions at a chosen moment — a single frame of a map replay.

In [ ]:
t = 8 * 3600  # 08:00
pts = [xy for trip in rec.trips() if (xy := rec.position_xy_at(trip, t, coord_map)) is not None]
lats = [p[0] for p in pts]
lons = [p[1] for p in pts]

fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(lons, lats, s=8, color="#d62728", alpha=0.7)
ax.set_title(f"Train positions at {format_hms(t)} ({len(pts)} trains)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect(1.6)
ax.grid(color="0.92")
plt.show()